In [ ]:
import os
from datetime import datetime

import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import random
import torch
import numpy as np

from models.convnext import load_model
from utils.infer_patch import infer_dataset
from utils.dataset import  get_glacioclim_dataset_path



In [ ]:
""" path """
# path of glacier_cham dataset 
cham_path = '/mnt/shared-storage/shared/Glaciomega/Glaciomega2_valdataset/glacier_cham'

# Filter bad images (clouds)
check_csv_file = 'data/image_glacioclim_filter.csv'

# path of glacioclim data
base_p_glacioclim = 'data/gdf_speed_glacioclim.gpkg'


"""model"""
# model weight path
model_type = 'multi' 
pretrain_ConvNeXtDPT= 'weights/convNext_base_DPT_finetune.pth' 
pretrain_DPTHeadTemporal = 'weights/convNext_base_DPT_multitemp_MultiRelativePos.pth' 
encoder_type = 'gla3'

""" seed """
seed = 42

""" device """
device = 'cuda'


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
seed_everything(seed)

In [ ]:
# load model
model_ConvNeXtDPT, temporal_head = load_model( model_type, encoder_type, pretrain_ConvNeXtDPT, pretrain_DPTHeadTemporal, use_weight_from_DPT=False, freeze_bbone=True, device=device )


In [ ]:
# load the dataset
cham_dataset = get_glacioclim_dataset_path( cham_path, check_csv_file )

In [ ]:
# inference
# refined : predicted DTM
# full_pred_depth : predicted correction on Glo30
# full_pred_var : predicted error
refined, full_pred_depth, full_pred_var = infer_dataset( cham_dataset, 
                                                         list(cham_dataset.keys())[0],
                                                         model_ConvNeXtDPT,
                                                         temporal_head, 
                                                         use_time_forward = True,
                                                         p_out=None, 
                                                         infer_size = 512, 
                                                         encoder_type=encoder_type, 
                                                         device=device, 
                                                         model_type = model_type)


In [ ]:
"""
Description of gdf_speed_glacioclim.gpkg

Result of the aggregation of velocity data available at https://glacioclim.osug.fr/Donnees-des-Alpes
Description of columns present in the raw data:
Columns common to all years:

stake_year_setup: year the stake was installed
stake_number: stake number
day_start, month_start, year_start: measurement start date
day_end, month_end, year_end: measurement end date
altitude_start: start altitude
altitude_end: end altitude
annual_speed: annual velocity

Acquisition-year-dependent columns:

x_lambert3_start, y_lambert3_start: start position in the Lambert 3 coordinate system
x_lambert3_end, y_lambert3_end: end position in the Lambert 3 coordinate system
x_lambert2_start, y_lambert2_start: start position in the Lambert 2 coordinate system
x_lambert2_end, y_lambert2_end: end position in the Lambert 2 coordinate system
profile_name: profile name

Added columns (not present in the raw dataset):

source_file: source of the raw data
year_file: end year of acquisition
x_start_2154, y_start_2154, x_end_2154, y_end_2154: start & end acquisition positions in the EPSG:2154 coordinate system
start_date, end_date: acquisition start / end date
acq_date_start, acq_date_end: nearest Sentinel-2 acquisition dates
altitude_start_3855, altitude_end_3855: start and end altitude in the GLO-30 vertical reference system (EPSG:3855)
x_start_repro, y_start_repro, x_end_repro, y_end_repro: pixel position within the Sentinel-2 image time series
"""

gdf_speed = gpd.read_file( base_p_glacioclim )

In [ ]:
####### Get all values at correct position/time

def get_existing_value( data, x, y ):
    if (x < 0) or (y < 0):
        return np.nan
    if (x > data.shape[0]) or (y > data.shape[1]):
        return np.nan
    x,y = int(x), int(y)
    return data[y,x]

cham_data_name = list(cham_dataset.keys())[0]
list_start_refined, list_end_refined, list_alti_start, list_alti_end = [],[], [],[]
list_start_sigma, list_end_sigma = [],[]
list_start_date, list_end_date = [], []
list_profile = []
list_ecart_start = []
list_ecart_end = []

for i, row in gdf_speed.iterrows():

    # date Sentinel to use
    acq_date_start = row['acq_date_start'].strftime('%Y%m%d')
    acq_date_end = row['acq_date_end'].strftime('%Y%m%d')

    # index 
    idx_start = cham_dataset[ cham_data_name ]['acq_date'].index(acq_date_start)
    idx_end = cham_dataset[ cham_data_name ]['acq_date'].index(acq_date_end)

    # x,y image position
    start_x_pxl, start_y_pxl = row['x_start_repro'],row['y_start_repro']
    end_x_pxl, end_y_pxl = row['x_end_repro'],row['y_end_repro']

    start_refined = get_existing_value( refined[ idx_start ], start_x_pxl, start_y_pxl )
    end_refined = get_existing_value( refined[ idx_end ], end_x_pxl, end_y_pxl )

    start_sigma = get_existing_value( full_pred_var[ idx_start ].squeeze(), start_x_pxl, start_y_pxl )
    end_sigma = get_existing_value( full_pred_var[ idx_end ].squeeze(), end_x_pxl, end_y_pxl )


    list_start_date.append( acq_date_start )
    list_end_date.append( acq_date_end )

    list_start_refined.append( start_refined )
    list_end_refined.append( end_refined )

    list_alti_start.append( row['altitude_start_3855'] )
    list_alti_end.append( row['altitude_end_3855'] )

    list_start_sigma.append( start_sigma )
    list_end_sigma.append( end_sigma )
    list_profile.append( row['profile_name'])

    list_ecart_start.append( (row['acq_date_start'] - row['start_date']).days )
    list_ecart_end.append( (row['acq_date_end'] - row['end_date']).days )

dates_start = pd.to_datetime(list_start_date, format="%Y%m%d")
dates_end   = pd.to_datetime(list_end_date,   format="%Y%m%d")
err_start = np.array(list_start_refined) - np.array(list_alti_start)
err_end   = np.array(list_end_refined)   - np.array(list_alti_end)

In [ ]:
# Total error
err_full = np.concatenate( [err_end, err_start])

print(" MAE:", np.nanmean(np.abs(err_full)))
print(" ME:", np.nanmean(err_full))
print(" RMSE:", np.sqrt(np.nanmean(err_full**2)))

In [ ]:
# Amplitude error
plt_alti_start     = np.array(list_alti_start)
plt_alti_end       = np.array(list_alti_end)
plt_start_refined  = np.array(list_start_refined)
plt_end_refined    = np.array(list_end_refined)

amp_gps =  plt_alti_start - plt_alti_end
amp_pred =  plt_start_refined - plt_end_refined
err_amp = amp_gps - amp_pred

print(" MAE:", np.nanmean(np.abs(err_amp)))
print(" ME:", np.nanmean(err_amp))
print(" RMSE:", np.sqrt(np.nanmean(err_amp**2)))

In [ ]:
# Error over time
fig, ax = plt.subplots(figsize=(12, 4))

ax.axhline(0, linestyle='--', color='#444444', lw=1.1, alpha=0.6, zorder=1)

sc = ax.scatter(dates_start, err_start, s=2.5, alpha=1,
                c=np.array(list_start_sigma), vmin=0, vmax=5,
                cmap='magma', rasterized=False, zorder=2)
ax.scatter(dates_end, err_end, s=2.5, alpha=1,
           c=np.array(list_end_sigma), vmin=0, vmax=5,
           cmap='magma', rasterized=False, zorder=2)

cb = fig.colorbar(sc, ax=ax, pad=0.01, fraction=0.025)
cb.set_label("Predicted uncertainty (m)", fontsize=9)
cb.ax.tick_params(labelsize=8)

ax.set_ylabel("Error (m)\n$\\it{Predicted - GPS}$", fontsize=10)
ax.tick_params(labelsize=9)
ax.tick_params(axis='x', rotation=30)

ax.grid(True, lw=0.4, alpha=0.3, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
sentinel_acq_date = pd.to_datetime(cham_dataset[cham_data_name]['acq_date'], format="%Y%m%d")
full_pred_var = full_pred_var.squeeze()
H, W = refined.shape[1], refined.shape[2]
cols = ['x_start_repro', 'y_start_repro', 'x_end_repro', 'y_end_repro']
gdf_speed_filter = gdf_speed.dropna(subset=cols).copy()

m = (
    (gdf_speed_filter['x_start_repro'] >= 0) &
    (gdf_speed_filter['y_start_repro'] >= 0) &
    (gdf_speed_filter['x_start_repro'] < W) &
    (gdf_speed_filter['y_start_repro'] < H) &
    (gdf_speed_filter['x_end_repro']   >= 0) &
    (gdf_speed_filter['y_end_repro']   >= 0) &
    (gdf_speed_filter['x_end_repro']   < W) &
    (gdf_speed_filter['y_end_repro']   < H) &
    (gdf_speed_filter['profile_name']  == 'Tour Noir') # comment to show other glacier
)
gdf_speed_filter = gdf_speed_filter[m].copy()

out = widgets.Output()
point_sel = widgets.SelectMultiple(description="Point", rows=6)
point_sel.options = range(len(gdf_speed_filter))

def update_plot(*args):
    with out:
        out.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(12, 5))

        for idx in point_sel.value:
            row = gdf_speed_filter.iloc[idx]
            x_start, y_start = int(row['x_start_repro']), int(row['y_start_repro'])
            x_end,   y_end   = int(row['x_end_repro']),   int(row['y_end_repro'])
            refined_pts_start = refined[:, y_start, x_start]
            var_pts_start     = full_pred_var[:, y_start, x_start]

            sc = ax.scatter(sentinel_acq_date, refined_pts_start,
                            s=18, c=var_pts_start, cmap='magma',
                            vmin=0, vmax=5, zorder=3, label=f'Prédit')

            ax.scatter(row['start_date'], row['altitude_start_3855'],
                       color='red', s=60, zorder=4, edgecolors='black',
                       label=f'Point GPS')
        cb = fig.colorbar(sc, ax=ax, pad=0.01, fraction=0.025)
        cb.set_label("Incertitude prédite (m)", fontsize=9)
        cb.ax.tick_params(labelsize=8)

        ax.set_ylabel("Altitude (m)",  fontsize=10)
        ax.set_title(row['profile_name'], fontsize=11, fontweight='600')
        ax.tick_params(labelsize=9)
        ax.tick_params(axis='x', rotation=30)
        ax.legend(frameon=True, framealpha=0.85,
                  edgecolor='#cccccc', fontsize=8.5)

        ax.grid(True, lw=0.4, alpha=0.3, color='gray')

        plt.tight_layout()
        plt.show()

point_sel.observe(update_plot, "value")

display(widgets.VBox([point_sel, out]))

In [ ]:
# mean by campaign
gt_ = []
pred_ = []

dates_start = pd.to_datetime(list_start_date, format="%Y%m%d")
dates_end   = pd.to_datetime(list_end_date, format="%Y%m%d")

plt_profile        = np.array(list_profile)
plt_dates_start = np.array(dates_start)
plt_dates_end   = np.array(dates_end)

plt_alti_start     = np.array(list_alti_start)
plt_alti_end       = np.array(list_alti_end)
plt_start_refined  = np.array(list_start_refined)
plt_end_refined    = np.array(list_end_refined)

df = pd.DataFrame({
    "start": plt_dates_start,
    "end": plt_dates_end,
    "alti_start": plt_alti_start,
    "alti_end": plt_alti_end,
    "start_refined": plt_start_refined,
    "end_refined": plt_end_refined,
})

df = df.sort_values("start")

df["campaign"] = (df["start"].diff() > pd.Timedelta(days=90)).cumsum()

campaign_stats = df.groupby("campaign").agg({
    "start": "mean",
    "end": "mean",
    "alti_start": "mean",
    "alti_end": "mean",
    "start_refined": "mean",
    "end_refined": "mean"
})



plt.figure(figsize=(12,5))

for _,row in campaign_stats.iterrows():

    gt_.append( row["alti_end"]-row["alti_start"] )
    pred_.append( row["end_refined"]-row["start_refined"] )

    plt.plot( [row["start"],row["end"]], [0, (row["alti_end"]-row["alti_start"])], linewidth=4.0, color='red')
    plt.plot( [row["start"],row["end"]], [0, (row["end_refined"]-row["start_refined"])], linewidth=4.0, color='blue')


gt_ = np.array(gt_)
pred_ = np.array(pred_)
rmse_all = np.sqrt( (((gt_ - pred_)**2).mean() ))

plt.title( f'rmse {round(rmse_all,2)}' )
plt.show()


In [ ]:
### error when taking mean of closest temporal values
def get_n_nearest_dates(date, date_index, n=3):
    """Return the n nearest dates from date_index to the given date."""
    diffs = pd.Series(date_index - date).abs()
    return date_index[diffs.nsmallest(n).index]

#date acq sentinel
df_acq_dates = pd.to_datetime(
    cham_dataset[cham_data_name]['acq_date'], format="%Y%m%d"
).sort_values()

# n nearest value to get
N_NEAREST = 3

list_start_refined_2, list_end_refined_2, list_alti_start_2, list_alti_end_2 = [],[], [],[]
list_start_sigma_2, list_end_sigma_2 = [],[]
list_start_date_2, list_end_date_2 = [], []
list_profile_2 = []
list_ecart_start_2 = []
list_ecart_end_2 = []

for i, row in gdf_speed.iterrows():

    nearest_starts = get_n_nearest_dates(row['start_date'], df_acq_dates, n=N_NEAREST)
    nearest_ends   = get_n_nearest_dates(row['end_date'],   df_acq_dates, n=N_NEAREST)
    
    # date d'acquisition Sentinel à utiliser
    acq_date_start = row['acq_date_start'].strftime('%Y%m%d')
    acq_date_end = row['acq_date_end'].strftime('%Y%m%d')
    
    start_refined_mean = 0
    end_refined_mean = 0
    start_sigma_mean = 0
    end_sigma_mean = 0

    for j in range( N_NEAREST ):
        acq_date_start = nearest_starts[ j ].strftime('%Y%m%d')
        acq_date_end = nearest_starts[ j ].strftime('%Y%m%d')
    
        # index des inférences associé
        idx_start = cham_dataset[ cham_data_name ]['acq_date'].index(acq_date_start)
        idx_end = cham_dataset[ cham_data_name ]['acq_date'].index(acq_date_end)

        # position x,y sur l'image
        start_x_pxl, start_y_pxl = row['x_start_repro'],row['y_start_repro']
        end_x_pxl, end_y_pxl = row['x_end_repro'],row['y_end_repro']

        start_refined = get_existing_value( refined[ idx_start ], start_x_pxl, start_y_pxl )
        end_refined = get_existing_value( refined[ idx_end ], end_x_pxl, end_y_pxl )

        start_sigma = get_existing_value( full_pred_var[ idx_start ].squeeze(), start_x_pxl, start_y_pxl )
        end_sigma = get_existing_value( full_pred_var[ idx_end ].squeeze(), end_x_pxl, end_y_pxl )

        start_refined_mean += start_refined
        end_refined_mean += end_refined
        start_sigma_mean += start_sigma
        end_sigma_mean += end_sigma
    
    start_refined_mean /= N_NEAREST
    end_refined_mean /= N_NEAREST
    start_sigma_mean /= N_NEAREST
    end_sigma_mean /= N_NEAREST

    list_start_date_2.append( acq_date_start )
    list_end_date_2.append( acq_date_end )

    list_start_refined_2.append( start_refined )
    list_end_refined_2.append( end_refined )

    list_alti_start_2.append( row['altitude_start_3855'] )
    list_alti_end_2.append( row['altitude_end_3855'] )

    list_start_sigma_2.append( start_sigma )
    list_end_sigma_2.append( end_sigma )
    list_profile_2.append( row['profile_name'])

    list_ecart_start_2.append( (row['acq_date_start'] - row['start_date']).days )
    list_ecart_end_2.append( (row['acq_date_end'] - row['end_date']).days )

# Erreur total sur l'amplitude
plt_alti_start     = np.array(list_alti_start_2)
plt_alti_end       = np.array(list_alti_end_2)
plt_start_refined  = np.array(list_start_refined_2)
plt_end_refined    = np.array(list_end_refined_2)

amp_gps =  plt_alti_start - plt_alti_end
amp_pred =  plt_start_refined - plt_end_refined
err_amp = amp_gps - amp_pred

print(" MAE:", np.nanmean(np.abs(err_amp)))
print(" ME:", np.nanmean(err_amp))
print(" RMSE:", np.sqrt(np.nanmean(err_amp**2)))

